# Document Question Answering System using RAG
**Week 7 Assignment:** Advanced RAG, LLM Internals, Fine-Tuning, Evaluation, ReAct Framework

In [1]:
# Install all required libraries
!pip install PyMuPDF sentence-transformers faiss-cpu transformers accelerate rank_bm25 groq -q

print("✅ All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.3 MB/s eta 0:00:00
✅ All libraries installed successfully!


##Step 1: Upload PDF
Run the cell below, click "Choose Files", and upload PDF.

In [2]:
from google.colab import files
import fitz  # PyMuPDF (for PDFs)
import os

# Upload the file
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"📄 Uploaded: {filename}")

# Extract text based on file type
def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.pdf':
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    else:  # Assume TXT or other text files
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()

raw_text = extract_text(filename)
print(f"✅ Extracted {len(raw_text)} characters.")

Saving Data Structures Full Notes.pdf to Data Structures Full Notes.pdf
📄 Uploaded: Data Structures Full Notes.pdf
✅ Extracted 151370 characters.


## Step 2: Advanced Chunking (With Overlap)
We use a chunk size of **1200** characters with **300** characters overlap. This prevents code snippets from being cut off and improves retrieval accuracy.

In [4]:
def chunk_text_with_overlap(text, chunk_size=1200, overlap=300):
    chunks = []
    start = 0
    text_length = len(text)
    while start < text_length:
        end = start + chunk_size
        if end > text_length:
            end = text_length
        chunks.append(text[start:end])
        if end == text_length:
            break
        start = end - overlap
    return chunks

chunks = chunk_text_with_overlap(raw_text)
print(f"✅ Created {len(chunks)} overlapping chunks.")

✅ Created 168 overlapping chunks.


## Step 3: Embeddings & Vector Database
We use the `all-mpnet-base-v2` model (better than MiniLM) and store vectors in FAISS.

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the embedding model
embed_model = SentenceTransformer('all-mpnet-base-v2')

print("⏳ Generating embeddings for all chunks... (takes 2-3 minutes)")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"✅ Vector database ready with {index.ntotal} vectors.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating embeddings for all chunks... (takes 2-3 minutes)


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Vector database ready with 168 vectors.


## Step 4: Hybrid Retrieval (Vector + BM25 Keyword Search)
We combine **semantic search** (embeddings) with **keyword search** (BM25) to catch both meaning and exact phrases. This ensures questions like "What is time complexity?" find the right chunk even if the embedding misses it.

In [6]:
from rank_bm25 import BM25Okapi
import numpy as np

# Build BM25 index from chunks
tokenized_chunks = [chunk.split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def retrieve_chunks_hybrid(query, top_k=20, alpha=0.5):
    """
    alpha: weight for vector search vs BM25 (0.5 = equal weight)
    """
    # 1. Vector search
    query_emb = embed_model.encode([query])
    distances, indices = index.search(np.array(query_emb), top_k)
    # Convert distances to similarity scores (closer = higher score)
    vector_scores = 1 - (distances[0] / np.max(distances[0] + 1e-8))

    # 2. BM25 search
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    # Get top_k BM25 indices
    bm25_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    # Normalize BM25 scores
    if np.max(bm25_scores) > 0:
        bm25_scores_norm = bm25_scores[bm25_indices] / np.max(bm25_scores)
    else:
        bm25_scores_norm = np.zeros(len(bm25_indices))

    # 3. Combine scores
    combined_scores = {}
    for i, idx in enumerate(indices[0]):
        combined_scores[idx] = alpha * vector_scores[i]
    for i, idx in enumerate(bm25_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + (1 - alpha) * bm25_scores_norm[i]

    # Sort by combined score
    sorted_indices = sorted(combined_scores.keys(), key=lambda i: combined_scores[i], reverse=True)

    # Return top_k unique chunks
    retrieved_texts = [chunks[idx] for idx in sorted_indices[:top_k]]
    return retrieved_texts

## Step 5: Groq Llama 3 API (Free, No Credit Card Required)
We use Groq's free API with Llama 3 70B for accurate information extraction. Sign up at console.groq.com.

In [7]:
from groq import Groq
import getpass

print("🔑 Get your free Groq API key from console.groq.com")
api_key = getpass.getpass("Paste your Groq API Key: ")
client = Groq(api_key=api_key)

print("✅ Groq client ready! Using Llama 3.3 70B")

🔑 Get your free Groq API key from console.groq.com
Paste your Groq API Key: ··········
✅ Groq client ready! Using Llama 3.3 70B


## Step 6: RAG Pipeline (Retrieve → Augment → Generate)
The `ask_question` function:
1. Retrieves the top 20 chunks using hybrid search (BM25 + vector).
2. Truncates context to fit the model's context window.
3. Instructs Llama 3 to extract information precisely.

In [8]:
def ask_question(question):
    # 1. Retrieve chunks using hybrid search
    retrieved_chunks = retrieve_chunks_hybrid(question, top_k=20)

    # 2. Remove duplicates
    unique_chunks = list(dict.fromkeys(retrieved_chunks))

    # 3. Build context with truncation (max ~4000 chars)
    context = ""
    for chunk in unique_chunks:
        if len(context) + len(chunk) > 4000:
            break
        context += chunk + "\n\n---\n\n"

    # 4. Generic prompt that works for any document
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.

RULES:
- Extract information EXACTLY as it appears in the context.
- If the context contains CODE, output that code exactly as written, inside a markdown code block.
- If the answer is NOT in the context, say "I don't have this information in the PDF."
- Keep the answer concise and direct.

Context:
{context}

Question: {question}

Answer:"""

    # 5. Generate with Groq
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=500
    )

    answer = completion.choices[0].message.content.strip()
    return answer, unique_chunks[:5]

## Step 7: Interactive Question Answering
Type any question about your document. Type `exit` to quit.

In [9]:
print("="*60)
print("🤖 RAG PIPELINE - Interactive Mode")
print("Using Groq Llama 3.3 70B for accurate extraction")
print("Type 'exit' to quit.")
print("="*60)

while True:
    user_question = input("\n❓ Your Question: ")
    if user_question.lower() in ['exit', 'quit', 'q']:
        print("\n👋 Goodbye!")
        break

    if user_question.strip() == "":
        print("⚠️ Please enter a valid question.")
        continue

    print("\n⏳ Searching and generating answer...")
    ans, sources = ask_question(user_question)

    print("\n" + "="*60)
    print(f"📝 Answer:\n{ans}")
    print("\n📄 Top Sources (Retrieved):")
    for i, src in enumerate(sources):
        snippet = src[:200].replace('\n', ' ').replace('  ', ' ')
        print(f"  {i+1}. {snippet}...")
    print("="*60)

🤖 RAG PIPELINE - Interactive Mode
Using Groq Llama 3.3 70B for accurate extraction
Type 'exit' to quit.

❓ Your Question: What is a data structure?

⏳ Searching and generating answer...

📝 Answer:
In computer science, a data structure is a way of organizing and storing data in a computer program so that it can be accessed and used efficiently.

📄 Top Sources (Retrieved):
  1.    1 Data Structures Unit-1 Introduction:  WHAT IS DATA STRUCTURE? In computer science, a data structure is a way of organizing and storing data in a computer program so that it can be access...
  2. and amount of data to be processed, the operations that need to be performed on the data and the efficiency requirements of the program. Efficient use of data structures can greatly improve the pe...
  3. the last node of the list holds the address of the first node hence forming a circular chain.  A Circular Linked List is a variation of the linked list where the last node points back to the first...
  4.    60 Data 

## Conclusion

### RAG System Architecture
PDF Upload → Text Extraction → Chunking (1200 chars, 300 overlap) →
Embeddings (all-mpnet-base-v2) → FAISS Vector DB →
Hybrid Search (BM25 + Vector, top_k=20) →
Groq Llama 3.3 70B → Answer

### Week 7 Topics Applied

| Topic | Implementation |
|-------|----------------|
| **Advanced RAG** | Chunk overlap, hybrid search (BM25 + FAISS), top_k=20 retrieval for better recall |
| **LLM Internals** | Used prompt engineering instead of fine-tuning to adapt the model |
| **Fine-Tuning (LoRA/PEFT)** | Discussed as a viable option for domain-specific datasets; prompt engineering works effectively for extraction tasks |
| **LLM Evaluation** | Manual validation by printing top 5 retrieved sources; confirmed retrieval quality before generation |
| **ReAct Framework** | Demonstrates "Act" (retrieval tool call) and "Reason" (generation); foundational for building autonomous agents |

### Model Selection Journey

| Model | Performance |
|-------|-------------|
| **FLAN-T5-Large** | Failed to extract structured information (email, phone) |
| **FLAN-T5-XL** | Still struggled with precise extraction |
| **Groq Llama 3.3 70B** | ✅ Perfect extraction and code generation |

### Testing on Data Structures PDF

| Question Type | Result |
|---------------|--------|
| Definitions (Data Structure, Stack, Recursion) | ✅ Correct definitions retrieved |
| Time Complexity Analysis | ✅ O(n²) and O(log n) explanations extracted |
| Code Extraction (Bubble Sort, Selection Sort, Stack Implementation) | ✅ Code output with proper formatting |
| LIFO/FIFO Concepts | ✅ Successfully explained |

### Key Takeaways
1. **Retrieval vs Generation:** Retrieval successfully found relevant chunks, but generation quality depended heavily on the model. Stronger LLMs extract information more reliably.
2. **Hybrid Search:** BM25 + Vector search significantly improved exact-match retrieval (e.g., "time complexity").
3. **Chunk Overlap:** Critical for preserving code snippets and context across chunk boundaries.
4. **Prompt Engineering:** Well-crafted prompts can adapt general-purpose LLMs to specific tasks without fine-tuning.

### Summary
This project demonstrates a robust RAG pipeline that answers questions from custom PDFs. The system successfully:
- Extracts definitions and explanations from technical documents
- Outputs code snippets exactly as written
- Handles both semantic and keyword-based queries

The combination of hybrid retrieval and a powerful LLM makes this solution scalable for any document type.

**Submitted BY:** Aditi Mehta